# Retail Sales CDAO — Monthly GCS Upload

Pulls the pre-landed monthly table(s) from `BI_SANDBOX` on `RSD-RBSQLDEV`,
writes them to CSV, and uploads to the `fmn-sandbox` bucket.

**How to use this notebook**

1. Edit the config cell below (only on first run, or when the month changes).
2. Run every cell top-to-bottom (`Runtime → Run All`, or `Shift+Enter` through each).
3. The "Pre-flight checks" cell will tell you *before* any data moves if something is misconfigured.

First-time setup (terminal, once): see `README.md`.

## 1. Config

This is the only cell you normally edit. Everything else just runs.

In [ ]:
# ── SQL Server ────────────────────────────────────────────────────────────────
SQL_SERVER   = "RSD-RBSQLDEV"
SQL_PORT     = 1433
SQL_DATABASE = "BI_SANDBOX"
SQL_SCHEMA   = "dbo"
SQL_DRIVER   = "ODBC Driver 17 for SQL Server"   # or 18 — whichever you installed

# ── Month + tables ────────────────────────────────────────────────────────────
STAMP  = "202601"                     # YYYYMM — change this each month
TABLES = [f"BASE{STAMP}"]             # e.g. [f"BASE{STAMP}", f"TRNS{STAMP}"] once TRNS is landed

# ── Google Cloud ──────────────────────────────────────────────────────────────
GCP_PROJECT = "fmn-sandbox"
GCP_BUCKET  = "customer_spend_data"   # or "customer_spend_data_processed"

# ── Local ─────────────────────────────────────────────────────────────────────
OUT_DIR = "./out"                     # where CSVs are cached before upload

# ── Safety switch ─────────────────────────────────────────────────────────────
# Set to True on the first run to verify CSVs without touching GCS.
DRY_RUN = False

## 2. Imports

If any of these fail, run the `pip install` command in the error message
from a terminal, then restart the kernel.

In [ ]:
import os, sys, time
from pathlib import Path

import pandas as pd
import pyodbc

from google.cloud import storage
from google.api_core import exceptions as gexc

print(f"Python {sys.version.split()[0]}  |  pandas {pd.__version__}  |  pyodbc {pyodbc.version}")

## 3. Pre-flight checks

Runs five quick checks before we touch any data. If all five are green,
the rest of the notebook will succeed.

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def _sql_connection_string():
    parts = [
        f"DRIVER={{{SQL_DRIVER}}}",
        f"SERVER={SQL_SERVER},{SQL_PORT}",
        f"DATABASE={SQL_DATABASE}",
        "Encrypt=yes",
        "TrustServerCertificate=yes",
    ]
    user, pw = os.getenv("SQL_USER"), os.getenv("SQL_PASSWORD")
    if user and pw:
        parts += [f"UID={user}", f"PWD={pw}"]
    else:
        parts.append("Trusted_Connection=yes")
    return ";".join(parts) + ";"


def sql_connect():
    drivers = [d for d in pyodbc.drivers() if "SQL Server" in d]
    if SQL_DRIVER not in drivers:
        raise RuntimeError(
            f"ODBC driver {SQL_DRIVER!r} not installed.\n"
            f"  Installed: {drivers or '(none)'}\n"
            f"  macOS:   brew tap microsoft/mssql-release && brew install msodbcsql17\n"
            f"  Linux:   sudo apt-get install unixodbc-dev msodbcsql17\n"
            f"  Windows: download 'Microsoft ODBC Driver 17 for SQL Server' from Microsoft\n"
            f"  Or set SQL_DRIVER above to one of the installed drivers."
        )
    return pyodbc.connect(_sql_connection_string(), timeout=15)


def table_exists(conn, table, schema=SQL_SCHEMA):
    cur = conn.cursor()
    cur.execute(
        "SELECT 1 FROM INFORMATION_SCHEMA.TABLES "
        "WHERE TABLE_SCHEMA = ? AND TABLE_NAME = ?",
        schema, table,
    )
    return cur.fetchone() is not None


def _adc_path():
    # gcloud's default ADC location — different on Windows
    if os.name == "nt":
        return Path(os.environ.get("APPDATA", "")) / "gcloud" / "application_default_credentials.json"
    return Path.home() / ".config" / "gcloud" / "application_default_credentials.json"


def resolve_gcp_credentials():
    """Returns a description of which credential method will be used."""
    gac = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
    if gac:
        if not Path(gac).is_file():
            raise RuntimeError(f"GOOGLE_APPLICATION_CREDENTIALS points to missing file: {gac}")
        return f"GOOGLE_APPLICATION_CREDENTIALS: {gac}"
    if _adc_path().is_file():
        return f"gcloud ADC: {_adc_path()}"
    raise RuntimeError(
        "No GCP credentials. In a terminal, run:\n"
        "    gcloud auth application-default login"
    )


def gcs_client():
    return storage.Client(project=GCP_PROJECT)


def check_bucket(client, bucket_name):
    try:
        bucket = client.get_bucket(bucket_name)
    except gexc.NotFound:
        raise RuntimeError(f"Bucket {bucket_name!r} not found in project {GCP_PROJECT!r}.")
    except gexc.Forbidden:
        raise RuntimeError(
            f"No access to {bucket_name!r}. Your account needs 'Storage Object Admin'."
        )
    probe = bucket.blob(".writecheck_retail_cdao")
    try:
        probe.upload_from_string("ok", timeout=10)
        probe.delete(timeout=10)
    except gexc.Forbidden:
        raise RuntimeError(
            f"Read-only on {bucket_name!r}. Needs objectCreator + objectDeleter (or Storage Object Admin)."
        )


# ── Run checks ────────────────────────────────────────────────────────────────

results = []

def _step(label, fn):
    try:
        detail = fn()
        results.append(("OK", label, detail or ""))
    except Exception as e:
        results.append(("FAIL", label, str(e)))


conn = None
creds_desc = None

def _sql_step():
    global conn
    conn = sql_connect()
    return f"{SQL_SERVER}:{SQL_PORT}/{SQL_DATABASE}"

def _tables_step():
    if conn is None:
        raise RuntimeError("skipped — SQL connection failed above")
    missing = [t for t in TABLES if not table_exists(conn, t)]
    if missing:
        raise RuntimeError(
            f"Not found: {missing}. Ask Sipho whether they've been landed in "
            f"{SQL_DATABASE}.{SQL_SCHEMA}."
        )
    return ", ".join(TABLES)

def _creds_step():
    global creds_desc
    creds_desc = resolve_gcp_credentials()
    return creds_desc

def _bucket_step():
    if creds_desc is None:
        raise RuntimeError("skipped — no GCP credentials")
    check_bucket(gcs_client(), GCP_BUCKET)
    return f"gs://{GCP_BUCKET} (write-check passed)"


_step("SQL Server reachable + auth", _sql_step)
_step("Expected tables exist",       _tables_step)
_step("GCP credentials resolvable",  _creds_step)
_step("GCS bucket exists + writable", _bucket_step)

print("\nPre-flight checks")
print("─" * 60)
for status, label, detail in results:
    mark = "✓" if status == "OK" else "✗"
    print(f"  [{mark}] {label}" + (f"  — {detail}" if detail and status == "OK" else ""))
    if status == "FAIL":
        for line in detail.splitlines():
            print(f"       {line}")
print("─" * 60)

all_ok = all(s == "OK" for s, _, _ in results)
if all_ok:
    print("All green — safe to continue.\n")
else:
    print("Fix the failures above before running the next cells.\n")

## 4. Extract tables → CSV

Streams each table in 50k-row chunks so memory stays flat even on a laptop.
Previews the first few rows and row-count so you can eyeball the data.

In [ ]:
def fetch_to_csv(conn, table, csv_path, chunksize=50_000):
    sql = f"SELECT * FROM [{SQL_SCHEMA}].[{table}]"
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    t0, total, first = time.time(), 0, True
    for chunk in pd.read_sql(sql, conn, chunksize=chunksize):
        chunk.to_csv(csv_path, index=False, mode="w" if first else "a", header=first)
        total += len(chunk)
        first = False
        print(f"  ... {total:,} rows", end="\r")

    print(f"  {total:,} rows in {time.time()-t0:.1f}s → {csv_path}")
    return total


assert all_ok, "Pre-flight checks did not pass — see the previous cell."

out_dir = Path(OUT_DIR).expanduser()
csv_paths = {}
csv_rowcounts = {}  # exact count captured while writing — no re-reading

with sql_connect() as conn:
    for table in TABLES:
        print(f"\n[{table}]")
        csv_paths[table] = out_dir / f"{table}.csv"
        csv_rowcounts[table] = fetch_to_csv(conn, table, csv_paths[table])

print("\nDone. CSVs saved to", out_dir.resolve())

### Preview

Quick sanity-check — shape and first few rows of each extract.

In [ ]:
for table, path in csv_paths.items():
    total_rows = csv_rowcounts.get(table, 0)
    if total_rows == 0:
        print(f"\n── {table} ── EMPTY (0 rows)")
        continue
    df_head = pd.read_csv(path, nrows=5)
    print(f"\n── {table} ── {total_rows:,} rows × {len(df_head.columns)} cols")
    display(df_head)

## 5. Upload to GCS

If `DRY_RUN = True` (set in the config cell) this cell will skip.

In [ ]:
if DRY_RUN:
    print("DRY_RUN is True — skipping upload.")
    print("CSVs are at:", [str(p) for p in csv_paths.values()])
else:
    client = gcs_client()
    bucket = client.bucket(GCP_BUCKET)

    for table, local_path in csv_paths.items():
        remote = local_path.name
        t0 = time.time()
        print(f"\nUploading {local_path.name} → gs://{GCP_BUCKET}/{remote} ...")
        bucket.blob(remote).upload_from_filename(str(local_path))
        size_mb = local_path.stat().st_size / 1e6
        print(f"  done — {size_mb:.1f} MB in {time.time()-t0:.1f}s")

    print("\n✓ Upload complete.")

## 6. Verify (optional)

Lists the files currently in the bucket so you can confirm the new uploads are there.

In [ ]:
if not DRY_RUN:
    client = gcs_client()
    print(f"Contents of gs://{GCP_BUCKET}/ :\n")
    blobs = sorted(client.list_blobs(GCP_BUCKET), key=lambda b: b.updated, reverse=True)
    for blob in blobs[:20]:
        print(f"  {blob.updated:%Y-%m-%d %H:%M}  {blob.size/1e6:>8.2f} MB   {blob.name}")